In [0]:
%pip install pymupdf

In [0]:
%restart_python

In [0]:
import importlib
import notebooks.file_ingestion as file_ingestion

importlib.reload(file_ingestion)

process_document = file_ingestion.process_document

In [0]:
from notebooks.file_ingestion import process_document

result = process_document(
    spark=spark,
    document_id="3e6a087f-5d9e-483a-a704-fedeb7563e0a"
)

print(result)

In [0]:
from notebooks.document_chunking import process_document

result = process_document(
    spark=spark,
    document_id="3e6a087f-5d9e-483a-a704-fedeb7563e0a"
)

print(result)

In [0]:
%pip install -U cohere

In [0]:
from config.config import (
    COHERE_EMBEDDING_MODEL,
    COHERE_SECRET_SCOPE,
    COHERE_SECRET_KEY
)

api_key = dbutils.secrets.get(
    scope=COHERE_SECRET_SCOPE,
    key=COHERE_SECRET_KEY
)

print("API key loaded:", bool(api_key))
print("Embedding model:", COHERE_EMBEDDING_MODEL)

In [0]:
%pip install transformers

In [0]:
%restart_python

In [0]:
from tokenization.local_tokenizer import LocalCohereTokenizer

tokenizer = LocalCohereTokenizer()

test_text = """

ЗАКОН О РАДУ | Član 3

Колективним уговором код послодавца, у складу са законом, уређују се права, обавезе и 
одговорности из радног односа и међусобни односи учесника колективног уговора.

Правилником о раду, односно уговором о раду, у складу са законом, уређују се права, обавезе 
и одговорности из радног односа:

1) ако код послодавца није основан синдикат или ниједан синдикат не испуњава услове 
репрезентативности или није закључен споразум о удруживању у складу са овим законом;

2) ако ниједан учесник колективног уговора не покрене иницијативу за почетак преговора ради 
закључивања колективног уговора;

3) ако учесници колективног уговора не постигну сагласност за закључивање колективног 
уговора у року од 60 дана од дана започињања преговора;

4) ако синдикат, у року од 15 дана од дана достављања позива за почетак преговора за 
закључивање колективног уговора, не прихвати иницијативу послодавца.

У случају из става 2. тачка 3) овог члана учесници колективног уговора дужни су да наставе 
преговоре у доброј вери.

У случају из става 2. тачка 3) овог члана, послодавац је дужан да правилник о раду достави 
репрезентативном синдикату у року од седам дана од дана његовог ступања на снагу.

Послодавац који не прихвати иницијативу репрезентативног синдиката за приступање 
преговорима за закључење колективног уговора, не може правилником о раду да уреди права 
и обавезе из радног односа.

Правилник о раду доноси надлежни орган код послодавца, утврђен законом, односно 
оснивачким или другим општим актом послодавца, а код послодавца који нема својство 
правног лица доноси овлашћено лице у складу са законом.

Правилник о раду јавног предузећа и друштва капитала чији је оснивач Република, аутономна 
покрајина или јединица локалне самоуправе (у даљем тексту: јавно предузеће) и друштва 
капитала чији је оснивач јавно предузеће, доноси се уз претходну сагласност оснивача.
"""

token_count = tokenizer.count_tokens(test_text)

print(f"Token count: {token_count}")

In [0]:
%reload_ext autoreload

In [0]:
from embedding.cohere_embedding_client import CohereEmbeddingClient

embedding_client = CohereEmbeddingClient()

embedding = embedding_client.generate_embeddings(["text", "text"])

print(str(len(embedding[0])) + " " + str(len(embedding[1])))

In [0]:
from embedding.cohere_embedding_client import CohereEmbeddingClient
from embedding.document_embedding import process_document

# Initialize Cohere client
embedding_client = CohereEmbeddingClient()

# Document selected for testing
document_id = "3e6a087f-5d9e-483a-a704-fedeb7563e0a"

# Generate and save embeddings
processed_count = process_document(
    spark=spark,
    document_id=document_id,
    embedding_client=embedding_client,
    batch_size=2
)

print(f"Generated embeddings: {processed_count}")

In [0]:
%pip install databricks-ai-search
%restart_python

In [0]:
from embedding.cohere_embedding_client import CohereEmbeddingClient
from embedding.query_embedding import generate_query_embedding

from retrieval.vector_search import (
    get_vector_search_index,
    search_chunks
)

embedding_client = CohereEmbeddingClient()
index = get_vector_search_index()

print("Clients initialized successfully.")

In [0]:
question = "Koliko dana godišnjeg odmora pripada zaposlenom?"

query_embedding = generate_query_embedding(
    question,
    embedding_client
)

print("Embedding dimensions:", len(query_embedding))
print("First 5 values:", query_embedding[:5])

In [0]:
results = search_chunks(
    index=index,
    question=question,
    query_embedding=query_embedding,
    num_results=5
)

print(f"Retrieved {len(results)} results.")
print(results)

In [0]:
from notebooks.document_retrieval import retrieve_context

result = retrieve_context(
    spark=spark,
    query="Koliko dana godišnjeg odmora pripada zaposlenom?",
    dbutils=dbutils
)

context = result["context"]
sources = result["sources"]

print(context)
print(sources)

In [0]:
%pip install databricks-openai

In [0]:
%restart_python

In [0]:
%pip install databricks-openai

In [0]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI(use_ai_gateway=True)

response = client.chat.completions.create(
    model="system.ai.qwen35-122b-a10b",
    messages=[
        {
            "role": "system",
            "content": "Odgovaraj na srpskom jeziku."
        },
        {
            "role": "user",
            "content": "Objasni ukratko šta je Zakon o radu."
        }
    ],
    max_tokens=300
)

print(response.choices[0].message.content)

In [0]:
from notebooks.document_generation import generate_answer

result = generate_answer(
    spark=spark,
    query="Ukoliko dam otkaz u junu, koliko dana godisnjeg odmora imam?",
    dbutils=dbutils
)

print("ODGOVOR:")
print(result["answer"])

print("\nIZVORI:")
for source in result["sources"]:
    print(
        f'{source["document_title"]} - '
        f'Član {source["article_number"]}'
    )